In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType
import pyspark.sql.functions as F

# The raw JSON text string from the Salesforce REST API
raw_json_text = """{
  "totalSize": 2,
  "done": true,
  "records": [
    {
      "Id": "0068W000014unXpQAI",
      "Name": "  ACME Cloud Deal  ",
      "Amount": 150000.00,
      "Account": {
        "Name": "ACME Corp",
        "Industry": "Technology"
      }
    },
    {
      "Id": "0068W000014unXpQAJ",
      "Name": "Globex Expansion",
      "Amount": null,
      "Account": null
    }
  ]
}"""

# Step 1: Define Account level schema
accountschema = StructType([
    StructField("Name", StringType(), True),
    StructField("Industry", StringType(), True)
])

# Step 2: Define Records array schema
recordsSchema = StructType([
    StructField("Id", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("Amount", DoubleType(), True),
    StructField("Account", accountschema, True)
])

# Step 3: FIXED - Changed StringType to StructType here
schema = StructType([
    StructField('totalSize', StringType(), True),
    StructField('done', StringType(), True),
    StructField('records', ArrayType(recordsSchema), True),
])

# Step 4: FIXED - Added a comma to the tuple (raw_json_text,) to create a valid 1-column row
df = spark.createDataFrame([(raw_json_text,)], ["payload_string_json"])

# Step 5: Parse JSON string using fixed schema
df_parsed = df.withColumn("parsed_data", F.from_json(F.col("payload_string_json"), schema))

df_exploded  = df_parsed.withColumn('records', F.explode(F.col('parsed_data.records')))

dff = df_exploded.select('records.*', F.col("records.Account.Name").alias("account_name"),
                        F.col("records.Account.Industry").alias("account_industry"))
dff = dff.drop(F.col('Account'))


dff = dff.withColumn('tName',F.ltrim(F.col("Name")))


# Displays the column list: ['payload_string_json', 'parsed_data']
display(dff)
